<a href="https://colab.research.google.com/github/broadinstitute/missense-pfes/blob/main/Copy_of_Compute_PFES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import sys
if 'google.colab' in sys.modules:
  %pip install g2papi
import g2papi


In [20]:
# @title # Input your gene/protein (HGNC symbol/UniProt accession) and a variant (e.g. M1V)
# @markdown Forms support many types of fields.

gene = 'UMOD'  # @param {type: "string"}
uniprot = 'P07911'  # @param {type: "string"}
variant = 'C77Y'  # @param {type: "string"}

# Input your protein (UniProt Accession) and a variant (e.g. M1V)
 - Import protein features from Genomics 2 Proteins portal via g2papi

In [21]:
# Get protein features as a pandas dataframe
protein_features = g2papi.get_protein_features(gene, uniprot)
protein_features.fillna('-', inplace=True)

protein_features

/tmp/ipykernel_3392149/125996553.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  protein_features.fillna('-', inplace=True)


,residueId,AA,Amino acid residues,Amino acid properties,Secondary structure (PDBe/SIFTS),Secondary structure (DSSP 3-state)*,Secondary structure (DSSP 9-state)*,Accessible surface area (Å²)*,Phi angle (degrees)*,Psi angle (degrees)*,...,Intra-chain Non-bonded interaction (PDB),Intra-chain Non-bonded interaction (AlphaFold2),Intra-chain Disulfide bond (PDB),Intra-chain Disulfide bond (AlphaFold2),Intra-chain Salt bridge (PDB),Intra-chain Salt bridge (AlphaFold2),Inter-chain Hydrogen bond (PDB),Inter-chain Non-bonded interaction (PDB),Inter-chain Disulfide bond (PDB),Inter-chain Salt bridge (PDB)
0,1,M,Methionine,Aliphatic,-,C (loop/coil),C (loop/coil),254,360.0,119.5,...,-,-,-,-,-,-,-,-,-,-
1,2,G,Glycine,"Special, lack of a chiral carbon, smallest ami...",-,C (loop/coil),C (loop/coil),79,-150.8,169.3,...,-,-,-,-,-,-,-,-,-,-
2,3,Q,Glutamine,Polar/Neutral,-,C (loop/coil),C (loop/coil),190,-81.1,164.4,...,-,-,-,-,-,-,-,-,-,-
3,4,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),123,-91.6,172.7,...,-,-,-,-,-,-,-,-,-,-
4,5,S,Serine,Polar/Neutral,-,C (loop/coil),C (loop/coil),118,-151.9,135.2,...,-,-,-,-,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,636,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),133,-92.0,123.9,...,-,-,-,-,-,-,-,-,-,-
636,637,L,Leucine,Aliphatic,-,C (loop/coil),C (loop/coil),141,-119.7,102.6,...,-,-,-,-,-,-,-,-,-,-
637,638,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),130,-98.4,123.2,...,-,-,-,-,-,-,-,-,-,-
638,639,F,Phenylalanine,Aromatic,-,C (loop/coil),C (loop/coil),184,-143.9,129.5,...,-,-,-,-,-,-,-,-,-,-


In [22]:
import requests
import pandas as pd
from io import StringIO

# Construct the public URL for the Google Cloud Storage object
bucket_name = 'g2p-portal'
file_path = 'portal_data/2026_q1_data/uniprot_metadata.tsv'
gcs_public_url = f'https://storage.googleapis.com/{bucket_name}/{file_path}'

try:
    response = requests.get(gcs_public_url)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

    file_content = response.text

    # Load the file_content into a pandas DataFrame
    df_uniprot_metadata = pd.read_csv(StringIO(file_content), sep='\t')

    # Extract UniProt ID and PANTHER_protein_class
    uniprot_panther_data = df_uniprot_metadata[['UniprotKB_Entry', 'PANTHER_protein_class']]

    # Filter the uniprot_panther_data DataFrame using the uid (assuming 'uid' is defined)
    panther_class_for_uid = uniprot_panther_data[uniprot_panther_data['UniprotKB_Entry'] == uniprot]['PANTHER_protein_class']

    # Check if a class was found and print it
    if not panther_class_for_uid.empty:
        print(f"PANTHER Protein Class for UniProt ID '{uniprot}':")
        protein_class = panther_class_for_uid.iloc[0]
        display(protein_class)
        
    else:
        print(f"No PANTHER Protein Class found for UniProt ID '{uniprot}'.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing GCS file: {e}")
    print("This might be due to the file/bucket not being publicly accessible or the path being incorrect.")

PANTHER Protein Class for UniProt ID 'P07911':


'transmembrane signal receptor'

In [24]:
enrichment_df

All                                       \
                             OR     CI_lo       CI_up        p_value   
feature                                                                
SS:B                   2.254768  2.004584    2.536176   1.091941e-42   
SS:E                   2.090225  2.033714    2.148307   0.000000e+00   
SS:G                   1.457269  1.376190    1.543125   1.174503e-37   
SS:H                   1.796355  1.761533    1.831865   0.000000e+00   
SS:I                   3.352512  2.929775    3.836245   1.086628e-75   
...                         ...       ...         ...            ...   
PTM:Modified residue   1.592947  1.407127    1.803306   2.601859e-13   
PPI:HB_inter           4.680014  4.397911    4.980212   0.000000e+00   
PPI:SB_inter           4.299103  3.815612    4.843860  1.850363e-147   
PPI:DS_inter          61.364665  8.435649  446.393875   1.782586e-15   
PPI:NB_inter           5.547793  5.242647    5.870700   0.000000e+00   

                                                           \
                            q_value n_case_yes n_ctrl_yes   
feature                                                     
SS:B                   1.653969e-42      688.0      470.0   
SS:E                   0.000000e+00    13002.0    10362.0   
SS:G                   1.728197e-37     2331.0     2474.0   
SS:H                   0.000000e+00    27537.0    27431.0   
SS:I                   1.929701e-75      674.0      310.0   
...                             ...        ...        ...   
PTM:Modified residue   3.152841e-13      511.0      493.0   
PPI:HB_inter           0.000000e+00     3995.0     1359.0   
PPI:SB_inter          4.143205e-147     1025.0      369.0   
PPI:DS_inter           2.212124e-15       40.0        1.0   
PPI:NB_inter           0.000000e+00     5448.0     1589.0   

                     DNA_metabolism_protein                       ...  \
                                         OR     CI_lo      CI_up  ...   
feature                                                           ...   
SS:B                               1.326845  0.601032   2.929158  ...   
SS:E                               2.565312  2.131706   3.087117  ...   
SS:G                               1.026006  0.704961   1.493257  ...   
SS:H                               1.999075  1.753873   2.278557  ...   
SS:I                               1.168171  0.498316   2.738470  ...   
...                                     ...       ...        ...  ...   
PTM:Modified residue               0.140023  0.018191   1.077812  ...   
PPI:HB_inter                       1.740365  1.255327   2.412812  ...   
PPI:SB_inter                       3.418833  1.904233   6.138124  ...   
PPI:DS_inter                            NaN  0.033436  85.008931  ...   
PPI:NB_inter                       1.324306  0.982820   1.784444  ...   

                        transporter                       unclassified  \
                            q_value n_case_yes n_ctrl_yes           OR   
feature                                                                  
SS:B                   6.514111e-02       81.0       40.0     2.109190   
SS:E                   1.798163e-03      830.0      499.0     2.601245   
SS:G                   3.233178e-09      468.0      207.0     1.352728   
SS:H                  6.188299e-204     6984.0     3162.0     1.624725   
SS:I                   4.339487e-16      236.0       56.0     4.177097   
...                             ...        ...        ...          ...   
PTM:Modified residue   3.673000e-01       26.0       25.0     1.645974   
PPI:HB_inter           9.161209e-54      440.0       51.0     6.071895   
PPI:SB_inter           8.688474e-12       96.0       12.0     6.417629   
PPI:DS_inter           5.626450e-01        2.0        0.0          inf   
PPI:NB_inter           4.557072e-84      789.0      114.0     6.310655   

                                                                          \
                         CI_

In [23]:
enrichment_df[protein_class]

KeyError: 'transmembrane signal receptor'

In [ ]:
import requests
import pandas as pd
from io import StringIO

# Raw URL for the CSV file on GitHub
github_csv_url = 'https://raw.githubusercontent.com/broadinstitute/missense-pfes/main/results/enrichment_OR_by_protein_class.csv'

try:
    response = requests.get(github_csv_url)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    # Read the content into a pandas DataFrame
    enrichment_df = pd.read_csv(StringIO(response.text), header=[0, 1], index_col=0)

    print("Successfully loaded the enrichment data. Here's a preview:")
    display(enrichment_df.head())

    # --- Extract odd ratio for a given protein class ---
    # Filter the DataFrame for the desired protein class
    odd_ratio_data = enrichment_df[enrichment_df['Protein Class'] == protein_class]

    if not odd_ratio_data.empty:
        print(f"\nOdd Ratio for '{protein_class}':")
        # Assuming 'Odd Ratio' is the column name for the odd ratio
        display(odd_ratio_data[['Protein Class', 'Odd Ratio']])
    else:
        print(f"\nNo data found for protein class: '{protein_class}'.")
        print("Please check the exact spelling of the protein class.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing the GitHub CSV file: {e}")
    print("Please ensure the URL is correct and the file is publicly accessible.")


Successfully loaded the enrichment data. Here's a preview:


All                                                             \
               OR     CI_lo     CI_up       p_value       q_value n_case_yes   
feature                                                                        
SS:B     2.254768  2.004584  2.536176  1.091941e-42  1.653969e-42      688.0   
SS:E     2.090225  2.033714  2.148307  0.000000e+00  0.000000e+00    13002.0   
SS:G     1.457269  1.376190  1.543125  1.174503e-37  1.728197e-37     2331.0   
SS:H     1.796355  1.761533  1.831865  0.000000e+00  0.000000e+00    27537.0   
SS:I     3.352512  2.929775  3.836245  1.086628e-75  1.929701e-75      674.0   

                   DNA_metabolism_protein                      ...  \
        n_ctrl_yes                     OR     CI_lo     CI_up  ...   
feature                                                        ...   
SS:B         470.0               1.326845  0.601032  2.929158  ...   
SS:E       10362.0               2.565312  2.131706  3.087117  ...   
SS:G        2474.0               1.026006  0.704961  1.493257  ...   
SS:H       27431.0               1.999075  1.753873  2.278557  ...   
SS:I         310.0               1.168171  0.498316  2.738470  ...   

           transporter                       unclassified                      \
               q_value n_case_yes n_ctrl_yes           OR     CI_lo     CI_up   
feature                                                                         
SS:B      6.514111e-02       81.0       40.0     2.109190  1.581982  2.812093   
SS:E      1.798163e-03      830.0      499.0     2.601245  2.441887  2.771002   
SS:G      3.233178e-09      468.0      207.0     1.352728  1.173738  1.559012   
SS:H     6.188299e-204     6984.0     3162.0     1.624725  1.553577  1.699132   
SS:I      4.339487e-16      236.0       56.0     4.177097  2.979905  5.855267   

                                                             
               p_value        q_value n_case_yes n_ctrl_yes  
feature                                                      
SS:B      7.856937e-07   1.123978e-06       81.0      110.0  
SS:E     3.497657e-184  2.001437e-183     1971.0     2372.0  
SS:G      4.348250e-05   5.669237e-05      286.0      607.0  
SS:H      2.600656e-97   8.640888e-97     3866.0     7614.0  
SS:I      1.090192e-16   1.936031e-16       83.0       57.0  

[5 rows x 147 columns]

KeyError: 'Protein Class'